In [ ]:
import pandas as pd
import numpy as np
from copy import deepcopy

import string
import re

import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm

#!pip install py_stringsimjoin
import py_stringsimjoin as ssj
#!pip install py_stringmatching
import py_stringmatching as sm


import networkx as nx
import matplotlib.pyplot as plt

<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
<a id="schema-matching"></a>
# Schema Matching

In [7]:
def sim_table(TableA:pd.DataFrame, TableB:pd.DataFrame):
    A = pd.DataFrame({"A": TableA.columns})
    B = pd.DataFrame({"B": TableB.columns})
    S = A.assign(key=1).merge(B.assign(key=1), on="key").drop("key", axis=1)
    return S

def random_sim_table(TableA:pd.DataFrame, TableB:pd.DataFrame):
    S = sim_table(TableA, TableB)
    S["sim"] = np.random.rand(len(S))
    return S

def to_sim_table(SimMatrix:pd.DataFrame):
    return SimMatrix.stack().reset_index(name="sim")

def to_sim_matrix(SimTable:pd.DataFrame):
    return SimTable.pivot(index="A", columns="B", values="sim") \
              .rename_axis(None, axis=1).rename_axis(None, axis=0)

def string_preprocess(s:str, char:str=string.punctuation, word:list=[]):
    if type(s) is str:
        s = s.lower()
        for c in char:
            s = s.replace(c, " ")
        for w in word:
            s = s.replace(w, " ")
    else:
        s = str(s)
    s = re.sub(" +", " ", s)
    return s.strip()

In [8]:
def levenshtein_sim(row:pd.Series):
    lev = sm.Levenshtein()
    return lev.get_sim_score(
            string_preprocess(row["A"]),
            string_preprocess(row["B"])
        )

def levenshtein_label_based_similarity(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(levenshtein_sim, axis=1)
    return C.sort_values("sim", ascending=False)

# Jaro
def jaro_sim(row:pd.Series):
    jaro = sm.Jaro()
    return jaro.get_sim_score(
            string_preprocess(row["A"]),
            string_preprocess(row["B"])
        )

def jaro_label_based_similarity(TableA:pd.DataFrame,TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(jaro_sim, axis=1)                   
    return C


# Jaccard
def jaccard_sim(row:pd.Series):
    jac=sm.Jaccard()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return jac.get_sim_score(
            tok.tokenize(string_preprocess(row["A"])),
            tok.tokenize(string_preprocess(row["B"]))
    )

def jaccard_label_based_similarity(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(jaccard_sim, axis=1)
    return C.sort_values("sim", ascending=False)

# OverlapCoefficient
def OC_sim(row:pd.Series):
    oc = sm.OverlapCoefficient()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return oc.get_sim_score(
            tok.tokenize(string_preprocess(row["A"])),
            tok.tokenize(string_preprocess(row["B"]))
    )

def OC_label_based_similarity(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(OC_sim, axis=1)
    return C.sort_values("sim", ascending=False)

# JaroWinkler
def JaroWinkler_sim(row:pd.Series):
    jw = sm.JaroWinkler()
    return jw.get_sim_score(
            string_preprocess(row["A"]),
            string_preprocess(row["B"])
        )

def JaroWinkler_label_based_similarity(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(JaroWinkler_sim, axis=1)
    return C.sort_values("sim", ascending=False)

# MongeElkan
#Secondary similarity function. This is expected to be a sequence-based similarity measure 
#(defaults to Jaro-Winkler similarity measure).
def MongeElkan_sim(row:pd.Series):
    me = sm.MongeElkan()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return me.get_raw_score(
            tok.tokenize(string_preprocess(row["A"])),
            tok.tokenize(string_preprocess(row["B"]))
    )


def MongeElkan_label_based_similarity(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(MongeElkan_sim, axis=1)
    return C.sort_values("sim", ascending=False)

In [9]:
def jaccard_sim_value(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame):
    j = sm.Jaccard()
    return j.get_raw_score(
            TableA[row["A"]].apply(string_preprocess).tolist(),
            TableB[row["B"]].apply(string_preprocess).tolist()
        )

def jaccard_value_overlap_sim(TableA:pd.DataFrame, TableB:pd.DataFrame):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(jaccard_sim_value, args=(TableA, TableB), axis=1)
    return C.sort_values("sim", ascending=False)

def generalized_sim_value(row:pd.Series, TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    j = sm.GeneralizedJaccard(
            sim_func=sm.Levenshtein().get_sim_score,
            threshold=threshold
        )
    return j.get_raw_score(
            TableA[row["A"]].apply(string_preprocess).tolist(),
            TableB[row["B"]].apply(string_preprocess).tolist()
        )

def generalized_value_overlap_sim(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(generalized_sim_value, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim", ascending=False)

In [10]:
def funzione_similarita_internaLEV(row:pd.Series): # Levenshtein
    lev = sm.Levenshtein()
    return lev.get_sim_score(
            string_preprocess(row["AX"]),
            string_preprocess(row["AY"])
        )

def extended_value_overlap_sim_LEV(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['AX']
    TY.columns=['AY']
    PCC = TX.drop_duplicates().assign(key=1).merge(TY.drop_duplicates().assign(key=1), on='key').drop(columns='key')
##    PCC = TX.drop_duplicates().merge(TY.drop_duplicates(), how='cross')
    PCC["SimJac"] = PCC.apply(funzione_similarita_internaLEV, axis=1)
    INTERSEZIONE =  PCC[PCC.SimJac>=threshold]
    SoloInAX=PCC.loc[~PCC['AX'].isin(INTERSEZIONE['AX'])][['AX']].drop_duplicates()
    SoloInAY=PCC.loc[~PCC['AY'].isin(INTERSEZIONE['AY'])][['AY']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

def value_overlap_extended_jaccard_LEV(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(extended_value_overlap_sim_LEV, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim",ascending=False)

def funzione_similarita_internaJaccard(row:pd.Series): # Jaccard
    jac=sm.Jaccard()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return jac.get_sim_score(
            tok.tokenize(string_preprocess(row["AX"])),
            tok.tokenize(string_preprocess(row["AY"])))

def extended_value_overlap_sim_JAC(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['AX']
    TY.columns=['AY']
    PCC = TX.drop_duplicates().assign(key=1).merge(TY.drop_duplicates().assign(key=1), on='key').drop(columns='key')
#    PCC = TX.drop_duplicates().merge(TY.drop_duplicates(), how='cross')
    PCC["SimJac"] = PCC.apply(funzione_similarita_internaJaccard, axis=1)
    INTERSEZIONE =  PCC[PCC.SimJac>=threshold]
    SoloInAX=PCC.loc[~PCC['AX'].isin(INTERSEZIONE['AX'])][['AX']].drop_duplicates()
    SoloInAY=PCC.loc[~PCC['AY'].isin(INTERSEZIONE['AY'])][['AY']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

def value_overlap_extended_jaccard_JAC(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(extended_value_overlap_sim_JAC, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim",ascending=False)

In [11]:
def sim__join(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['AX']
    TY.columns=['AY']
    
    INTERSEZIONE  = ssj.jaccard_join(     TX, TY, # tabelle su cui effettuare il sim join
                                'AX', 'AY', # chiavi delle tabelle 
                                'AX', 'AY', # attributi di join
                                  sm.WhitespaceTokenizer(return_set=True),
                                  threshold=threshold, 
                                  show_progress=False,
                                  l_out_attrs=['AX'],  r_out_attrs=['AY']
                           )
    SoloInAX=TX.loc[~TX['AX'].isin(INTERSEZIONE['l_AX'])][['AX']].drop_duplicates()
    SoloInAY=TY.loc[~TY['AY'].isin(INTERSEZIONE['r_AY'])][['AY']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))


def value_overlap_simjoin_jaccard(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(sim__join, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim",ascending=False)

In [12]:
def max_sim_table(SimTableList:list):
    ST = pd.DataFrame(columns=["A","B","sim"])
    for x in SimTableList:
        ST = ST.append(x, ignore_index=True)
    return ST.groupby(["A","B"])["sim"].max().reset_index()

def min_sim_table(SimTableList:list):
    ST = pd.DataFrame(columns=["A","B","sim"])
    for x in SimTableList:
        ST = ST.append(x, ignore_index=True)
    return ST.groupby(["A","B"])["sim"].min().reset_index()

def avg_sim_table(SimTableList:list):
    ST = pd.DataFrame(columns=["A","B",'sim'])
    for x in SimTableList:
        ST = ST.append(x, ignore_index=True)
    return ST.groupby(["A","B"])["sim"].mean().reset_index()

In [13]:
def thresholding(SimTable:pd.DataFrame, threshold:float):
    return SimTable[SimTable["sim"] > threshold].sort_values(["sim"], ascending=[False])
    
def top_K(SimTable:pd.DataFrame, K:int, AoB:str="A"):
    MT = deepcopy(SimTable)
    MT["pos"] = MT.sort_values(["sim"], ascending=[False]).groupby(AoB).cumcount()
    return MT[MT["pos"] < K].drop(columns=["pos"]).sort_values([AoB,"sim"], ascending=[True,False])

def top_1(SimTable:pd.DataFrame, AoB:str="A"):
    return top_K(SimTable, 1, AoB)

In [14]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=["A", "B", "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT["A"].isin(MATCH["A"])) & (~MT["B"].isin(MATCH["B"]))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['A']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['B']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [15]:
def  Weighted_sum(dataframes, weights):
    if len(dataframes) < 2 or len(dataframes) != len(weights):
        raise ValueError("È necessario fornire almeno due DataFrame e i relativi pesi.")

    result_df = dataframes[0].copy()

    result_df["sim"] = result_df["sim"] * weights[0]

    for i in range(1, len(dataframes)):
        df = dataframes[i]
        weight = weights[i]

        result_df = pd.merge(result_df, df, on=["A", "B"], how="outer")

        result_df["sim"] = result_df.apply(lambda row: row["sim_x"] + row["sim_y"] * weight if not pd.isnull(row["sim_x"]) and not pd.isnull(row["sim_y"]) else row["sim_x"] if not pd.isnull(row["sim_x"]) else row["sim_y"] * weight, axis=1)

        result_df = result_df.drop(columns=["sim_x", "sim_y"])

    return result_df

In [16]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold.iloc[:, :2].copy()
    Match = Match.iloc[:, :2].copy()
    Gold.columns = Match.columns = ['A', 'B']

    FOJ = Gold.merge(Match, how='outer', indicator=True) # full outer join between Gold and Match

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)], 
                'F':[round(F,4)]
            })

def Vedi_Valuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold.iloc[:, :2].copy()
    Match = Match.iloc[:, :2].copy()
    Gold.columns = Match.columns = ['A', 'B']
    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN
    


def AnalisiGlobalMatchTable(GMT, Sources):
    """
    Verifica che la GMT sia coerente con le sorgenti. Stabilisce il tipo di mapping
    """
    # 1) Verifica che tutte le SOURCE usate in ['SOURCE'] siano presenti in Sources
    sources_in_GMT = set(GMT['SOURCE'])
    defined_sources = set(Sources.keys())
    print("1) Le seguenti SOURCE di GMT non sono definite in Sources:" , sorted(sources_in_GMT - defined_sources))

    # 2) Verifica che tutte le gli attributi locali GMT  siano presenti in Sources
    all_slat = set([f"{source}_{col}" for source, df in Sources.items() for col in df.columns])
    slat_in_GMT = set(GMT['SLAT'])
    print("2) I seguenti SLAT di GMT non sono definiti in Sources:" , sorted(slat_in_GMT - all_slat))

    # 3) GAT mappati in una sola SOURCE
    gat_source_counts = GMT.groupby('GAT')['SOURCE'].nunique()
    single_source_gats = gat_source_counts[gat_source_counts == 1].index.tolist()
    print("3) GAT mappati da una sola SOURCE:", sorted(single_source_gats))
    
    # 4) Per ogni SOURCE: GAT → più LAT
    print("4) GAT mappati in più LAT (per ciascuna SOURCE):")
    grouped = GMT.groupby(['SOURCE', 'GAT'])['LAT'].nunique()
    for (source, gat), count in grouped.items():
        if count > 1:
            print(f"   SOURCE: {source}, GAT: {gat}, LAT diversi: {count}")

    # 5) Per ogni SOURCE: LAT → più GAT
    print("5) LAT mappati in più GAT (per ciascuna SOURCE):")
    grouped = GMT.groupby(['SOURCE', 'LAT'])['GAT'].nunique()
    for (source, lat), count in grouped.items():
        if count > 1:
            print(f"   SOURCE: {source}, LAT: {lat}, GAT diversi: {count}")

In [17]:
def to_GMT(GMM:pd.DataFrame):
    """
    Converte una Global Matching Matrix (GMM) in una Global Matching Table (GMT),
    elencando per ogni attributo globale (GAT) gli attributi locali (LAT) associati 
    nelle varie sorgenti (SOURCE), con identificatore univoco SLAT = SOURCE_LAT.
    """
    GMT = pd.DataFrame(columns=['GAT', 'SOURCE', 'LAT', 'SLAT'])

    for x in GMM.index:
        for y in GMM.columns:
            for z in GMM.loc[x, y]:
                GMT.loc[len(GMT)] = [x, y, z, f"{y}_{z}"]

    return GMT

In [18]:
def to_GMM(GMT: pd.DataFrame):
    """
    Converte una Global Matching Table (GMT) in una Global Matching Matrix (GMM),
    raggruppando gli attributi locali (LAT) per attributo globale (GAT) e sorgente (SOURCE).
    """
    df = GMT.groupby(['GAT', 'SOURCE'])['LAT'].agg(list).unstack('SOURCE')
    for c in df.columns:
        df.loc[df[c].isnull(), [c]] = df.loc[df[c].isnull(), c].apply(lambda x: [])
    return df

In [19]:
def generaLAT(Sources):
  LAT = pd.DataFrame(columns=['SOURCE', 'LAT', 'SLAT'])
  for x in Sources:
    for y in Sources[x].columns:
      LAT.loc[len(LAT)]=[str(x),str(y), str(x)+'_'+str(y)]
  return LAT

In [20]:
def ClusterComponentiConnessi(LMT, SOURCES):

    MatchTable = LMT.iloc[:, :2].copy()
    MatchTable.columns = ['SLAT_A', 'SLAT_B']
    

    TuttiInodi= generaLAT(SOURCES)['SLAT'].to_list()
    Singleton = set(TuttiInodi) - set(MatchTable['SLAT_A']).union(set(MatchTable['SLAT_B']))


    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['SLAT_A'], row['SLAT_B'])

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [21]:
def SchemaIntegration(Sources):

    LMT = CalcoloLocalMatchingTable(Sources)[['SLAT_A','SLAT_B']]
    
    GMT = GMTComponentiConnessi(LMT,Sources)
    

    return to_GMM(GMT)

In [22]:
def MatchIndottiGMT(GMT):
  Join=pd.merge(GMT,GMT, on='GAT')
  Join=Join[Join.SOURCE_x<Join.SOURCE_y]
  Join=Join[['SLAT_x','SLAT_y']]
  Join.columns=['SLAT_A','SLAT_B']

  return Join.drop_duplicates()

<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
<a id="esercizio-top-down-camera"></a>
# Esercizio Top-Down : Camera

In [23]:
src_links = [
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS1_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS2_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS3_B.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/CameraS4_B.csv',
]
SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }
GlobalSchema=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/GlobalClass_Bis.csv').astype(str)
GoldStandard=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/TopDCamera/GoldStandardFull.csv').astype(str)
print(GlobalSchema.columns.to_list())
to_GMM(GoldStandard)

['brand', 'image_format', 'auto_focus_beam', 'auto_focus_mode', 'exposure_mode', 'battery_chemistry']


SOURCE,S1,S2,S3,S4
GAT,,,,
auto_focus_beam,[af assist beam],[],[],[af illuminator]
auto_focus_mode,"[autofocus af, af modes, focusarea selection]",[auto focusing af modes],"[auto focus, focus]",[autofocus]
battery_chemistry,[batteries],"[battery technology, battery type]",[battery type],[battery]
brand,[],[manufacturer],[brand],"[brand, product name]"
exposure_mode,[exposure control],"[light exposure modes, light exposure control]",[],[exposure modes]
image_format,"[file formats, still image type]",[image formats supported],[image format],[image format]


In [25]:
AnalisiGlobalMatchTable(GMT=GoldStandard, Sources=SOURCES)

1) Le seguenti SOURCE di GMT non sono definite in Sources: []
2) I seguenti SLAT di GMT non sono definiti in Sources: ['S2_manufacturer']
3) GAT mappati da una sola SOURCE: []
4) GAT mappati in più LAT (per ciascuna SOURCE):
   SOURCE: S1, GAT: auto_focus_mode, LAT diversi: 3
   SOURCE: S1, GAT: image_format, LAT diversi: 2
   SOURCE: S2, GAT: battery_chemistry, LAT diversi: 2
   SOURCE: S2, GAT: exposure_mode, LAT diversi: 2
   SOURCE: S3, GAT: auto_focus_mode, LAT diversi: 2
   SOURCE: S4, GAT: brand, LAT diversi: 2
5) LAT mappati in più GAT (per ciascuna SOURCE):


In [26]:
ValutazioneMatchTable = pd.DataFrame(columns=['MT', 'TP', 'FP', 'FN', 'P', 'R', 'F'])

In [27]:
def CalcoloMatchingTable(TableL:pd.DataFrame,TableR:pd.DataFrame):

# Matching Methods: Qui si definiscono i Base Matcher da utilizzare; se ne possono aggiungere altri    
        SimTableA = levenshtein_label_based_similarity(TableL, TableR)
        SimTableB = jaro_label_based_similarity(TableL, TableR)
        # SimTableB = vo.value_overlap_sim(GlobalSchema, Sources[y])
        SimTableC=  value_overlap_simjoin_jaccard(TableL, TableR, 0.5)

# Combined Approaches: qui si definisce il combiner; se ne possono aggiungere altri    
        #SimTable = avg_sim_table([SimTableA,SimTableB, SimTableC])
        #SimTable = min_sim_table([SimTableA,SimTableB, SimTableC])
        SimTable = max_sim_table([SimTableA,SimTableB, SimTableC])

        # Weighted-sum
        #SimTable = Weighted_sum([SimTableA,SimTableB,SimTableC], [.3,.4,.3] )
        
# Generating Correspondences: dalla tabella di similarità alle corrispondenze

### Local Single Attribute Strategies:
        MatchTable= thresholding(SimTable, 0.2)

        #MatchTable = lsa.top_K(SimTable,1,'A')
        MatchTable = top_K(MatchTable,1,'B')

        
### Global Mapping (si usano solo questi due metodi)
        #MatchTable = stable_marriage(MatchTable)
        #MatchTable = simmetric_best_match(MatchTable)

        return MatchTable

In [29]:
def CalcoloGlobalMatchingTable(Sources, GlobalSchema:pd.DataFrame):
    GlobalMatchingTable = pd.DataFrame(columns=['GAT','SOURCE','LAT','SLAT','sim'])
    for y in tqdm(Sources.keys()):
        MatchTable = CalcoloMatchingTable(GlobalSchema, Sources[y])
        
        MatchTable.columns = ['GAT','LAT','sim']
        MatchTable['SOURCE'] = str(y)
        MatchTable['SLAT'] = MatchTable['SOURCE']+'_'+MatchTable['LAT']
        GlobalMatchingTable = GlobalMatchingTable.append(MatchTable, sort=False)

    return GlobalMatchingTable

In [ ]:
GMTcalcolata=CalcoloGlobalMatchingTable(SOURCES, GlobalSchema)
to_GMM(GMTcalcolata)

100%|██████████| 4/4 [00:02<00:00,  1.80it/s]


SOURCE,S1,S2,S3,S4
GAT,,,,
auto_focus_beam,[af assist beam],[],[],[af illuminator]
auto_focus_mode,"[af modes, autofocus af, focusarea selection]",[auto focusing af modes],"[auto focus, focus]",[autofocus]
battery_chemistry,[batteries],"[battery technology, battery type]",[battery type],[battery]
brand,[],[],[brand],"[brand, product name]"
exposure_mode,[exposure control],"[light exposure control, light exposure modes]",[],[exposure modes]
image_format,"[file formats, still image type]",[image formats supported],[image format],[image format]


In [ ]:
X=Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']])
X

,MT,TP,FP,FN,P,R,F
0,26,26,0,1,1.0,0.963,0.9811


In [33]:
ValutazioneMatchTable = ValutazioneMatchTable.append(X).rename(index={0: "Label-Instance-Max-0.4-Top1-NoGlobal11"})
ValutazioneMatchTable

,MT,TP,FP,FN,P,R,F
Label-Instance-Avg-0.4-Top1-NoGlobal11,26,26,0,1,1.0000,0.9630,0.9811
Label-Instance-Avg-0.4-Top1-NoGlobal11,24,19,5,8,0.7917,0.7037,0.7451
Label-Instance-Max-0.4-Top1-NoGlobal11,26,26,0,1,1.0000,0.9630,0.9811
